# Sync Hugging Face Dataset to Kaggle Dataset V2

This notebook downloads all 16 video partition archives (`part_01.zip` through `part_16.zip`) and metadata from the private Hugging Face repository (branch: `version-2`) and creates a complete Kaggle Dataset: `kietkolat/hierarchical-harmful-video-dataset-v2`.

**Why run in Google Colab?**
- Colab provides **107 GB free disk space** (sufficient for the ~37 GB archive payload).
- Datacenter-to-datacenter speeds (AWS/GCP to Kaggle): whole process runs in **5-8 minutes** with **0 bytes uploaded from your home internet**.

In [ ]:
# Step 1: Install Hugging Face Hub and Kaggle CLI
!pip install -q huggingface_hub kaggle

In [ ]:
import os
import json
import shutil
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

# =============================================================================
# Step 2: Configure Credentials & Parameters
# =============================================================================
# Hugging Face Access Token (read permission is sufficient)
HF_TOKEN = "YOUR_HF_TOKEN_HERE"

# Kaggle API Credentials
os.environ["KAGGLE_USERNAME"] = "kietkolat"
os.environ["KAGGLE_KEY"] = "YOUR_KAGGLE_API_KEY_HERE"

# Hugging Face Source
HF_REPO_ID = "KietTuannn/hierarchical-harmful-video-archive"
HF_BRANCH = "version-2"

# Kaggle Destination
KAGGLE_DATASET_SLUG = "hierarchical-harmful-video-dataset-v2"
KAGGLE_DATASET_TITLE = "Hierarchical Harmful Video Dataset V2"
DATASET_DIR = Path("/content/kaggle_dataset")

print(f"Target Kaggle Dataset: {os.environ['KAGGLE_USERNAME']}/{KAGGLE_DATASET_SLUG}")

In [ ]:
# Step 3: Initialize Staging Directory & Metadata JSON
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

metadata = {
    "title": KAGGLE_DATASET_TITLE,
    "id": f"{os.environ['KAGGLE_USERNAME']}/{KAGGLE_DATASET_SLUG}",
    "licenses": [{"name": "CC0-1.0"}]
}
with open(DATASET_DIR / "dataset-metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Initialized dataset-metadata.json:")
print(json.dumps(metadata, indent=2))

In [ ]:
# Step 4: Scan available files from Hugging Face branch
api = HfApi(token=HF_TOKEN)
files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type="dataset", revision=HF_BRANCH)

video_files = sorted([f for f in files if f.startswith("videos/") and f.endswith(".zip")])
metadata_files = [f for f in files if f.startswith("metadata/")]

print(f"Discovered {len(video_files)} video part archives:")
for vf in video_files:
    print(f"  - {vf}")
print(f"Discovered {len(metadata_files)} metadata files: {metadata_files}")

In [ ]:
# Step 5: Download Metadata Files (label-origin.csv, etc.)
for remote_path in metadata_files:
    fname = Path(remote_path).name
    print(f"Downloading metadata: {remote_path}...")
    local_tmp = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=remote_path,
        revision=HF_BRANCH,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    shutil.copy2(local_tmp, DATASET_DIR / fname)
    print(f"Saved: {DATASET_DIR / fname}")

# Step 6: Download All Video Partition Archives
for idx, remote_zip in enumerate(video_files, start=1):
    fname = Path(remote_zip).name
    print(f"\n[{idx}/{len(video_files)}] Downloading {remote_zip}...")
    local_tmp = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=remote_zip,
        revision=HF_BRANCH,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    shutil.copy2(local_tmp, DATASET_DIR / fname)
    size_gb = (DATASET_DIR / fname).stat().st_size / (1024**3)
    print(f"Saved {fname} ({size_gb:.2f} GB)")

total_size = sum(f.stat().st_size for f in DATASET_DIR.glob("*") if f.is_file()) / (1024**3)
print(f"\nSUCCESS: Staged {len(list(DATASET_DIR.glob('*')))} files, Total size: {total_size:.2f} GB")

In [ ]:
# Step 7: Create Kaggle Dataset
!kaggle datasets create -p /content/kaggle_dataset --dir-mode zip

In [ ]:
# Step 8: Check Dataset Status on Kaggle
!kaggle datasets status kietkolat/hierarchical-harmful-video-dataset-v2